# Azure ML + Azure DevOps: MLOps CI/CD for Iris

**This is Notebook 2.** Notebook 1 (`aml_iris_end_to_end.ipynb`) showed how to *author* the
pipeline interactively with the **Python SDK v2** — clicking through cells, watching it run.

That is how you *experiment*. It is **not** how you ship to production. In production you don't
run notebooks by hand — a **git push** does it for you, every time, the same way, with tests and
approvals. That is **MLOps**, and this notebook builds it with **Azure DevOps Pipelines**.

```
NOTEBOOK 1  (SDK v2)                     NOTEBOOK 2  (CLI v2 + Azure DevOps)
═══════════════════                     ══════════════════════════════════
You click "Run" on each cell      →     git push triggers everything
MLClient(...).jobs.create(...)    →     az ml job create --file pipeline.yml
Good for exploring                →     Good for repeatable production
```

## What we build here

```
  git push to main
        │
        ▼
┌───────────────────────────────────────────────────────────────────┐
│  Azure DevOps Pipeline  (azure-pipelines.yml)                       │
│                                                                     │
│   STAGE 1  CI       lint + pytest + `az ml job validate`            │
│      │              (runs on every Pull Request — gates the merge)  │
│      ▼                                                              │
│   STAGE 2  Train    register env + data, run training pipeline,     │
│      │              register the model         (dev workspace)      │
│      ▼                                                              │
│   STAGE 3  Deploy   managed online endpoint + blue/green deploy     │
│                     ⏸  waits for human APPROVAL   (prod)            │
└───────────────────────────────────────────────────────────────────┘
        │
        ▼
   Live REST endpoint  →  POST /score  →  {"setosa" | "versicolor" | "virginica"}
```

> **Source of truth:** [MLOps — DevOps for machine learning](https://learn.microsoft.com/azure/machine-learning/overview-what-is-azure-machine-learning?view=azureml-api-2) ·
> [Azure DevOps for CI/CD](https://learn.microsoft.com/azure/machine-learning/how-to-devops-machine-learning?view=azureml-api-2)

---
## 1 — What is MLOps, really?

Microsoft defines it in one line:

> **"MLOps is DevOps for machine learning — a process for developing models for production.
A model's lifecycle from training to deployment must be auditable, if not reproducible."**

So MLOps is just **classic DevOps** (CI/CD, version control, automated tests, gated releases)
applied to the three things that make ML different from normal software:

| Normal software | ...plus ML adds |
|-----------------|------------------|
| **code** | **data** (which version trained this model?) |
| build artifact | **model** (registered + versioned in the Model Registry) |
| unit tests | **model quality gates** (accuracy/AUC must clear a bar) |

The features in Azure ML that make this possible (straight from the overview doc):

- **Git integration** — every job records the commit it ran from
- **MLflow integration** — params, metrics and the model are logged automatically
- **Pipeline scheduling** + **Event Grid triggers**
- **Ease of use with CI/CD tools like GitHub Actions or Azure DevOps**  ← *that's this notebook*

### CI vs CD in one sentence each

- **CI (Continuous Integration):** every change is automatically **linted, tested, and validated** before it can merge. → *Stage 1.*
- **CD (Continuous Delivery):** a merged, validated change is automatically **trained and deployed** through environments (dev → prod), with approvals. → *Stages 2 & 3.*

---
## 2 — Why CLI v2, not the SDK, for CI/CD

Notebook 1 used the **Python SDK v2** (`MLClient`, `@dsl.pipeline`). This notebook uses the
**Azure CLI v2** (`az ml ...`) for everything that runs inside the pipeline. Microsoft's own
guidance:

> *"The command-line based CLI is more convenient in CI/CD MLOps scenarios, while the SDK
> might be more convenient for development."*

Why the CLI wins for automation:

| | SDK v2 (Notebook 1) | CLI v2 (this notebook) |
|---|---|---|
| Assets defined as | Python objects | **declarative YAML files** (git-diffable) |
| Runs from | a Python kernel | **any shell** — the build agent |
| One command | several lines of Python | `az ml job create --file pipeline.yml` |
| Best for | exploring, debugging | **reproducible pipelines** |

> ⚠️ **CLI v1 was retired on 2025-09-30.** Everything here is **v2** (`az extension add -n ml`).
> The same Python *scripts* in `components/` run in **both** worlds — only the orchestration
> layer changes from SDK to YAML.

Install locally (one time) if you want to run the `az ml` cells below:
```bash
az extension add -n ml      # the v2 ML extension
az login
```

---
## 3 — The Azure DevOps building blocks

Four concepts do all the work. Learn these and the YAML reads itself:

| Concept | What it is | In this project |
|---|---|---|
| **Service connection** | how the pipeline authenticates to Azure | an **Azure Resource Manager** connection named `aml-arm-connection` |
| **Variable group** | shared, named values (Pipelines → Library) | `iris-mlops-vars` → `resourceGroup`, `workspace`, `location` |
| **Environment** | a named deploy target you can put **approvals/checks** on | `iris-prod` (require a human to click *Approve*) |
| **Stages / jobs / steps** | the structure of `azure-pipelines.yml` | `CI → Train → Deploy` |

```
azure-pipelines.yml
├── trigger:        when to run        (push / PR to main)
├── variables:      from variable group + literals
└── stages:
    ├── CI       → job → steps (bash, pytest, AzureCLI@2)
    ├── Train    → job → steps (AzureCLI@2: az ml job create)
    └── Deploy   → deployment job → environment: iris-prod  (⏸ approval)
```

The modern pattern (per the docs) is deliberately simple: an **ARM service connection** + the
built-in **`AzureCLI@2`** task running **`az ml`** commands. You do *not* need any special
marketplace task to train or deploy — just the CLI. (The optional *Azure ML* DevOps extension
only adds a convenience "wait for job" task.)

---
## 4 — Files this notebook adds to the repo

Notebook 1's `components/`, `dependencies/` and `data/` are **reused unchanged** — that's the
whole point: one set of scripts, two ways to run them. The new files are the *declarative* layer:

```
azue_ml_iris_project_devops/
├── aml_iris_end_to_end.ipynb     # NB1 — author with the SDK (experiment)
├── aml_iris_devops.ipynb         # NB2 — you are here (operationalize)
│
├── azure-pipelines.yml           # ★ the CI/CD pipeline (3 stages)
│
├── mlops/                        # ★ CLI v2 YAML assets the pipeline submits
│   ├── environment.yml           #   env  (reuses dependencies/conda.yml)
│   ├── data-asset.yml            #   data (reuses data/iris.csv)
│   ├── train-pipeline.yml        #   the prep→train pipeline (YAML twin of NB1)
│   ├── endpoint.yml              #   managed online endpoint
│   └── deployment.yml            #   model → endpoint deployment
│
├── tests/                        # ★ fast, cloud-free unit tests for the CI stage
│   └── test_pipeline_smoke.py
│
├── components/                   # reused from NB1 (data_prep.py, train.py, train.yml)
├── dependencies/conda.yml        # reused from NB1
└── data/iris.csv                 # reused from NB1
```

---
## 5 — Asset YAML #1 & #2: environment + data

In Notebook 1 you registered these with Python (`ml_client.environments.create_or_update(...)`).
Here they are **declarative YAML** — the exact same result, but version-controlled and
submitted by the pipeline with `az ml environment create` / `az ml data create`.

Run the next cell to print them.

In [ ]:
# Just display the YAML the pipeline will register — nothing runs in Azure here.
for f in ["mlops/environment.yml", "mlops/data-asset.yml"]:
    print(f"\n===== {f} " + "=" * (60 - len(f)))
    print(open(f).read())

---
## 6 — Asset YAML #3: the training pipeline

This is the **YAML twin of the `@dsl.pipeline` function** from Notebook 1. Same two steps,
same scripts, same wiring (`data_prep` outputs feed `train` inputs) — expressed as YAML so a
shell can submit it:

```
inputs.input_data ──► data_prep_job ──(train_data / test_data)──► train_job ──► registered model
```

Key lines to notice when you print it below:
- `code: ../components/data_prep` — **reuses Notebook 1's scripts**, no copies
- `${{parent.jobs.data_prep_job.outputs.train_data}}` — how one step feeds the next
- `path: azureml:iris-data@latest` — resolves to the data asset the Train stage just registered
- `registered_model_name: iris_defaults_model` — same model name as NB1, so both share one registry

In [ ]:
print(open("mlops/train-pipeline.yml").read())

---
## 7 — The CI quality gate: tests run *before* Azure

The single biggest MLOps habit: **fail fast, locally, for free.** Before the pipeline spends a
minute of cloud compute, the CI stage runs unit tests on the build agent. `tests/test_pipeline_smoke.py`
checks the data schema, guards against missing values, and trains the real estimator to confirm it
clears a baseline accuracy.

Run them right now — exactly what the CI stage runs:

In [ ]:
!python -m pytest tests -q

And the cloud-side CI check — `az ml job validate` parses the pipeline YAML and verifies every
asset reference and input/output wiring **against your workspace, without running anything**.

> Requires `az login` + `az extension add -n ml`. Uncomment to run; it's also Step 1 of the pipeline.

In [ ]:
# !az configure --defaults group=rg-aml-iris workspace=aml-iris-ws
# !az ml job validate --file mlops/train-pipeline.yml

---
## 8 — Deploy assets: endpoint + deployment

Same **endpoint vs deployment** split you learned in Notebook 1, now as YAML:

- **`endpoint.yml`** — the stable URL + auth. Created once, reused forever.
- **`deployment.yml`** — attaches a *specific model version* to that URL. The CD stage gives each
  run a unique name (`blue-$(Build.BuildId)`) so a new model rolls out **next to** the old one,
  gets smoke-tested, and only *then* takes 100% of traffic — **blue/green, zero downtime**.

Because `train.py` used `mlflow.sklearn.log_model()`, **no scoring script is needed** — MLflow
supplies the `predict()` interface automatically.

In [ ]:
for f in ["mlops/endpoint.yml", "mlops/deployment.yml"]:
    print(f"\n===== {f} " + "=" * (60 - len(f)))
    print(open(f).read())

---
## 9 — The pipeline itself: `azure-pipelines.yml`

This ties it all together. Read it top-to-bottom — the comments explain every block. The shape:

```
trigger:   push or PR to main
variables: - group: iris-mlops-vars      ← resourceGroup / workspace / location
           - serviceConnection, modelName, endpointName
stages:
  CI      ─ job validate ─ UsePythonVersion → pip install → flake8 → pytest → az ml job validate
  Train   ─ job train    ─ AzureCLI@2:  az ml compute/environment/data create
          │                            az ml job create --file mlops/train-pipeline.yml
          │                            az ml job stream + status check
          └                            publish MODEL_VERSION for the next stage
  Deploy  ─ deployment   ─ environment: iris-prod   ⏸ APPROVAL
                         ─ AzureCLI@2:  az ml online-endpoint create (if new)
                                        az ml online-deployment create --all-traffic
                                        az ml online-endpoint invoke   (smoke test)
                                        az ml online-endpoint update --traffic
```

Things worth internalizing as you read it:
- **`AzureCLI@2`** with `azureSubscription: $(serviceConnection)` is what authenticates each
  block to Azure — no keys in the YAML.
- **`az configure --defaults group=... workspace=...`** sets the target once per step, so the
  `az ml` commands stay short.
- **`##vso[task.setvariable ...;isOutput=true]`** is how the Train stage hands the new model
  version to the Deploy stage.
- The **`deployment` job + `environment: iris-prod`** is where a human approval gate lives.

In [ ]:
print(open("azure-pipelines.yml").read())

---
## 10 — One-time setup in Azure DevOps

The YAML is in your repo; now wire up Azure DevOps once. ([Full walkthrough](https://learn.microsoft.com/azure/machine-learning/how-to-devops-machine-learning?view=azureml-api-2))

**1. Project + repo** — create a project at <https://dev.azure.com>, push this folder to its repo
(Azure Repos, or connect GitHub).

**2. Service connection** (auth to Azure)
- *Project settings → Service connections → New → **Azure Resource Manager*** → Next
- Keep the default identity type, pick your subscription + resource group
- Name it **`aml-arm-connection`** (must match `serviceConnection` in the YAML)

**3. Variable group** (Pipelines → Library → **+ Variable group**)
- Name it **`iris-mlops-vars`** and add:
  | Variable | Example |
  |---|---|
  | `resourceGroup` | `rg-aml-iris` |
  | `workspace` | `aml-iris-ws` |
  | `location` | `germanywestcentral` |

**4. Environment + approval** (Pipelines → Environments → **New environment** → `iris-prod`)
- Open it → *Approvals and checks* → **+ Approvals** → add yourself.
- Now Stage 3 (Deploy) pauses until someone clicks **Approve** — your production gate.

**5. Create the pipeline** (Pipelines → **New pipeline**)
- Select your repo → *Existing Azure Pipelines YAML file* → `/azure-pipelines.yml` → **Run**.

That's it. From now on a `git push` to `main` runs CI → Train → (approve) → Deploy automatically.

---
## 11 — (Optional) Run the same steps by hand with the CLI

The pipeline just runs `az ml` commands — so you can run them yourself to understand each stage
before handing them to Azure DevOps. This is the best way to learn what the YAML actually does.

> Requires `az login`, `az extension add -n ml`, and an existing workspace. Uncomment to use.

In [ ]:
# # point the CLI at your workspace (same values as your variable group)
# !az configure --defaults group=rg-aml-iris workspace=aml-iris-ws location=germanywestcentral

# # --- STAGE 2 (Train), step by step ---
# !az ml environment create --file mlops/environment.yml
# !az ml data create --file mlops/data-asset.yml
# !az ml job create --file mlops/train-pipeline.yml --stream
# !az ml model list -n iris_defaults_model -o table

In [ ]:
# # --- STAGE 3 (Deploy), step by step ---
# !az ml online-endpoint create --file mlops/endpoint.yml
# !az ml online-deployment create --endpoint-name iris-endpoint-mlops --file mlops/deployment.yml --all-traffic
# !az ml online-endpoint invoke -n iris-endpoint-mlops --request-file sample/request.json

# # cleanup — online endpoints bill per hour while they exist
# !az ml online-endpoint delete -n iris-endpoint-mlops --yes --no-wait

---
## 12 — Summary: what you learned

| MLOps concept | Where it lives in this project |
|---|---|
| **Declarative assets** (env, data, pipeline) | `mlops/*.yml` — git-diffable, the CLI v2 way |
| **CI: fail fast & free** | `tests/` + `pytest` + `az ml job validate` in Stage 1 |
| **CD: train → register → deploy** | Stages 2 & 3 of `azure-pipelines.yml` |
| **Authentication** | ARM service connection `aml-arm-connection` (no keys in YAML) |
| **Portable config** | variable group `iris-mlops-vars` |
| **Production gate** | `environment: iris-prod` + manual approval |
| **Safe rollout** | blue/green deployment names + traffic shift |
| **Reproducibility** | every job records its git commit + MLflow run |

### How the two notebooks fit together

```
NB1  aml_iris_end_to_end.ipynb   →  EXPLORE   (SDK v2, run cells by hand)
NB2  aml_iris_devops.ipynb       →  SHIP      (CLI v2 + Azure DevOps, git push)
                                     ▲
                          same components/ + conda.yml + data/
```

### Where to go next
- Add a **model quality gate** stage: fail the build if `test_accuracy` drops below a threshold
  (read the metric back with `az ml job show` / the MLflow API).
- Add a **second variable group + environment** for a real `dev → qa → prod` promotion chain.
- Swap the manual approval for **automated checks** (e.g. an Azure Function gate).
- Schedule periodic **retraining** with an Event Grid or scheduled trigger.

> Docs that back this notebook:
> [What is Azure ML](https://learn.microsoft.com/azure/machine-learning/overview-what-is-azure-machine-learning?view=azureml-api-2) ·
> [Azure DevOps for CI/CD](https://learn.microsoft.com/azure/machine-learning/how-to-devops-machine-learning?view=azureml-api-2) ·
> [CLI v2 pipeline YAML schema](https://learn.microsoft.com/azure/machine-learning/reference-yaml-job-pipeline?view=azureml-api-2)